# Lekcija 21: Optički tok i procjena gibanja

Optički tok procjenjuje prividno gibanje između dvaju kadrova videozapisa: za svaki piksel (ili odabrani skup točaka) daje 2D vektor $(u, v)$ koji opisuje kamo se pomaknuo. Ova lekcija izvodi klasičnu metodu **Lucas-Kanade** iz istog tenzora strukture koji se koristio za detekciju kutova u lekciji 20, razmatra temeljni **problem otvora**, uspoređuje je s globalno optimiziranom metodom **Horn-Schunck** i završava gustim tokom pomoću metode **Farnebäck**.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Pretpostavka stalnosti svjetline

Optički tok pretpostavlja da se intenzitet točke ne mijenja tijekom gibanja: $I(x, y, t) = I(x+u, y+v, t+1)$. Taylorov razvoj desne strane prvog reda daje **jednadžbu ograničenja optičkog toka**:

$$I_x u + I_y v + I_t = 0$$

gdje su $I_x, I_y$ prostorni gradijenti (lekcija 12), a $I_t$ razlika intenziteta između kadrova. To je **jedna jednadžba s dvjema nepoznanicama** (gibanjem $u$ i $v$) za svaki pojedini piksel &mdash; sama po sebi ne daje dovoljno informacija za određivanje toka.

## Problem otvora

Kada kroz mali prozor promatramo rub koji se giba, vidljivo je samo gibanje **okomito na rub**; gibanje **duž ruba** ne stvara nikakvu vidljivu promjenu pa ga lokalno mjerenje ne može uočiti. Upravo je zato jednadžba ograničenja toka nedovoljno određena: ograničava samo komponentu $(u,v)$ duž smjera gradijenta $(I_x, I_y)$, dok okomita komponenta tom jednom jednadžbom ostaje potpuno neograničena.

In [ ]:
size = 160
rect_topleft = (50, 60)
rect_w, rect_h = 60, 40

def make_rect_frame(dx=0, dy=0, value=200):
    img = np.zeros((size, size), dtype=np.uint8)
    x0, y0 = rect_topleft[0] + dx, rect_topleft[1] + dy
    cv2.rectangle(img, (x0, y0), (x0 + rect_w, y0 + rect_h), value, -1)
    return img

true_motion = (8, 6)           # right and down
vertical_only_motion = (0, 6)  # same downward shift, but no horizontal component

frame_true = make_rect_frame(*true_motion)
frame_vertical = make_rect_frame(*vertical_only_motion)

# A small aperture window straddling the rectangle's TOP (horizontal) edge, centered well
# away from any corner and comfortably inside the canvas, so there are no border effects.
win_half = 10
wx, wy = rect_topleft[0] + rect_w // 2, rect_topleft[1]

def with_window(img_gray, color=(255, 0, 0)):
    vis = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)
    cv2.rectangle(vis, (wx - win_half, wy - win_half), (wx + win_half, wy + win_half), color, 1)
    return vis

fig, axes = plt.subplots(1, 2, figsize=(7, 4))
axes[0].imshow(with_window(frame_true))
axes[0].set_title(f'True motion (u,v)={true_motion}\n(right + down)', fontsize=9)
axes[1].imshow(with_window(frame_vertical))
axes[1].set_title(f'Vertical-only motion (u,v)={vertical_only_motion}\n(down only)', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

window_true = frame_true[wy - win_half:wy + win_half, wx - win_half:wx + win_half]
window_vertical = frame_vertical[wy - win_half:wy + win_half, wx - win_half:wx + win_half]
print('inside the red window, the two (different!) motions look pixel-for-pixel identical:',
      np.array_equal(window_true, window_vertical))

Dva para kadrova zapravo prikazuju različito globalno gibanje &mdash; jedan pomak udesno i prema dolje, a drugi samo prema dolje &mdash; no unutar crvenog prozora (koji obuhvaća gornji, potpuno vodoravni rub pravokutnika) ne mogu se razlikovati. Vodoravna komponenta ide *duž* tog ruba pa lokalno ne ostavlja nikakav trag; vidljiva je samo zajednička okomita komponenta, *okomita* na rub. Lokalno mjerenje u tom prozoru doista ne može razlikovati ta dva gibanja.

## Lucas-Kanade: rješavanje problema otvora pomoću prozora

<a href="../references.html#lucas-kanade-1981">Rješenje Lucasa i Kanadea (1981.)</a>: pretpostaviti da je tok $(u,v)$ **stalan unutar malog prozora**, a zatim kombinirati jednadžbu ograničenja toka svakog piksela u prozoru u preodređeni sustav najmanjih kvadrata:

$$\underbrace{\begin{bmatrix}\sum I_x^2 & \sum I_xI_y \\ \sum I_xI_y & \sum I_y^2\end{bmatrix}}_{M}\begin{bmatrix}u\\v\end{bmatrix} = -\begin{bmatrix}\sum I_xI_t\\ \sum I_yI_t\end{bmatrix}$$

$M$ je upravo tenzor strukture iz lekcije 20! To nije slučajno: za rješavanje sustava $M$ mora biti invertibilan, odnosno imati dvije velike svojstvene vrijednosti &mdash; upravo Shi-Tomasijev uvjet „dobre značajke za praćenje”. Ravnomjerno područje ($M$ blizu nule) ili rub (jedna mala svojstvena vrijednost &mdash; ponovno problem otvora) daju loše uvjetovan ili singularan sustav; kut daje dobro uvjetovan sustav. **Kutovi se mogu pratiti upravo iz istog razloga zbog kojeg su dobri kutovi.**

In [ ]:
rng = np.random.default_rng(0)
frame1 = np.zeros((200, 200), dtype=np.uint8)
for _ in range(20):
    x, y = rng.integers(20, 180, 2)
    radius = rng.integers(5, 15)
    cv2.circle(frame1, (x, y), radius, int(rng.integers(100, 255)), -1)

small_motion = (0.6, 0.4)  # sub-pixel motion, well inside the linear (Taylor) approximation's validity
shift = np.float32([[1, 0, small_motion[0]], [0, 1, small_motion[1]]])
frame2_small = cv2.warpAffine(frame1, shift, (200, 200))

Ix = cv2.Sobel(frame1.astype(np.float64), cv2.CV_64F, 1, 0, ksize=3, scale=1 / 8)
Iy = cv2.Sobel(frame1.astype(np.float64), cv2.CV_64F, 0, 1, ksize=3, scale=1 / 8)
It = frame2_small.astype(np.float64) - frame1.astype(np.float64)


def lucas_kanade_at(x, y, half_win=7):
    x, y = int(round(x)), int(round(y))
    ix = Ix[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    iy = Iy[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    it = It[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    A = np.stack([ix, iy], axis=1)
    solution, *_ = np.linalg.lstsq(A, -it, rcond=None)
    return solution

corners = cv2.goodFeaturesToTrack(frame1, maxCorners=30, qualityLevel=0.1, minDistance=10)
flows = np.array([lucas_kanade_at(p[0][0], p[0][1]) for p in corners])

print(f'true motion:        {small_motion}')
print(f'manual LK estimate: ({flows[:, 0].mean():.3f}, {flows[:, 1].mean():.3f})  (averaged over {len(corners)} corners)')

### Vizualizacija slike, praćenih značajki i njihova gibanja

In [ ]:
arrow_scale = 15  # true motion here is sub-pixel, so exaggerate the arrows to make them visible

vis_small = cv2.cvtColor(frame1, cv2.COLOR_GRAY2RGB)
for (x, y), (u, v) in zip(corners[:, 0], flows):
    p0 = (int(round(x)), int(round(y)))
    p1 = (int(round(x + u * arrow_scale)), int(round(y + v * arrow_scale)))
    cv2.arrowedLine(vis_small, p0, p1, (255, 0, 0), 1, tipLength=0.3)
    cv2.circle(vis_small, p0, 2, (0, 255, 0), -1)

plt.imshow(vis_small)
plt.title(f'Corners (green) and estimated motion (red, {arrow_scale}x exaggerated)\ntrue motion = {small_motion}')
plt.axis('off')
plt.show()

### Veća gibanja zahtijevaju iteraciju

Ovo jednokratno linearno rješavanje oslanja se na Taylorovu aproksimaciju, koja vrijedi samo za mala gibanja. Pri većem pomaku isti jednokratni pristup daje lošije rezultate:

In [ ]:
large_motion = (4.0, 3.0)
shift_large = np.float32([[1, 0, large_motion[0]], [0, 1, large_motion[1]]])
frame2_large = cv2.warpAffine(frame1, shift_large, (200, 200))

It_large = frame2_large.astype(np.float64) - frame1.astype(np.float64)

def lucas_kanade_large(x, y, half_win=7):
    x, y = int(round(x)), int(round(y))
    ix = Ix[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    iy = Iy[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    it = It_large[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    A = np.stack([ix, iy], axis=1)
    solution, *_ = np.linalg.lstsq(A, -it, rcond=None)
    return solution

flows_large_manual = np.array([lucas_kanade_large(p[0][0], p[0][1]) for p in corners])

next_pts, status, _ = cv2.calcOpticalFlowPyrLK(frame1, frame2_large, corners, None, winSize=(15, 15), maxLevel=2)
cv_flow = (next_pts - corners).reshape(-1, 2)[status.ravel() == 1]

print(f'true motion:                          {large_motion}')
print(f'single-shot manual LK estimate:       ({flows_large_manual[:, 0].mean():.3f}, {flows_large_manual[:, 1].mean():.3f})  <- degraded')
print(f'cv2.calcOpticalFlowPyrLK estimate:    ({cv_flow[:, 0].mean():.3f}, {cv_flow[:, 1].mean():.3f})  <- accurate')

OpenCV-jev `calcOpticalFlowPyrLK` obrađuje velika gibanja pokretanjem metode Lucas-Kanade **iterativno** (ponovnim preslikavanjem i linearizacijom do konvergencije) na **piramidi slika** (lekcija 11) &mdash; najprije grubo procjenjuje gibanje na maloj, zamućenoj verziji slike, a zatim ga usavršava razinu po razinu. Ta kombinacija omogućuje točno određivanje velikih gibanja iako osnovna linearna aproksimacija vrijedi samo lokalno, za pojedini mali korak.

### Vizualizacija vektora rijetkog toka

In [ ]:
arrow_scale = 3  # exaggerate the arrows a bit so they're easier to see

vis = cv2.cvtColor(frame1, cv2.COLOR_GRAY2RGB)
for (p0,), (p1,), ok in zip(corners, next_pts, status.ravel()):
    if not ok:
        continue
    p0 = p0.astype(int)
    p1_exaggerated = np.round(p0 + (p1 - p0) * arrow_scale).astype(int)
    cv2.arrowedLine(vis, tuple(p0), tuple(p1_exaggerated), (255, 0, 0), 1, tipLength=0.3)
    cv2.circle(vis, tuple(p0), 2, (0, 255, 0), -1)

plt.imshow(vis)
plt.title(f'Tracked corners, true motion = {large_motion} (arrows {arrow_scale}x exaggerated)')
plt.axis('off')
plt.show()

## Horn-Schunck: globalna alternativa toku u prozorima

Prozor metode Lucas-Kanade predstavlja *lokalnu* pretpostavku glatkoće: tok je stalan unutar malog susjedstva i procjenjuje se neovisno za svaku točku. Iste su godine <a href="../references.html#horn-schunck-1981">Horn i Schunck (1981.)</a> predložili *globalnu* alternativu: umjesto brojnih neovisnih rješavanja po prozorima, minimizirati jednu energiju preko cijele slike odjednom, uz kompromis između zadovoljavanja jednadžbe ograničenja optičkog toka i glatkoće promjene toka između susjednih piksela:

$$E(u,v) = \sum_{x,y} \underbrace{(I_x u + I_y v + I_t)^2}_{\text{data term}} \;+\; \alpha^2 \underbrace{\left(\|\nabla u\|^2 + \|\nabla v\|^2\right)}_{\text{smoothness term}}$$

Član glatkoće čini ovaj pristup globalnim: povezuje tok svakog piksela s tokovima njegovih susjeda, pa minimizacija $E$ (npr. iterativnim Gauss-Seidelovim ažuriranjima) omogućuje da se pouzdane procjene toka uz rubove *prošire* u ravnomjerna područja bez teksture koja nemaju vlastite lokalne informacije &mdash; upravo najizraženiji način neuspjeha koji stvara problem otvora.

In [ ]:
hs_img1 = np.zeros((120, 120), dtype=np.uint8)
cv2.rectangle(hs_img1, (40, 40), (80, 80), 200, -1)  # one textured square, otherwise flat
hs_true_motion = (3.0, 2.0)
hs_shift = np.float32([[1, 0, hs_true_motion[0]], [0, 1, hs_true_motion[1]]])
hs_img2 = cv2.warpAffine(hs_img1, hs_shift, (120, 120))

def horn_schunck(I1, I2, alpha=5.0, n_iters=100):
    Ix = cv2.Sobel(I1.astype(np.float64), cv2.CV_64F, 1, 0, ksize=3, scale=1 / 8)
    Iy = cv2.Sobel(I1.astype(np.float64), cv2.CV_64F, 0, 1, ksize=3, scale=1 / 8)
    It = I2.astype(np.float64) - I1.astype(np.float64)
    u, v = np.zeros_like(Ix), np.zeros_like(Ix)
    neighbor_avg = np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=np.float64) / 4.0
    for _ in range(n_iters):
        u_avg, v_avg = cv2.filter2D(u, -1, neighbor_avg), cv2.filter2D(v, -1, neighbor_avg)
        correction = (Ix * u_avg + Iy * v_avg + It) / (alpha ** 2 + Ix ** 2 + Iy ** 2)
        u, v = u_avg - Ix * correction, v_avg - Iy * correction
    return u, v

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, n_iters in zip(axes, [1, 20, 200]):
    u, v = horn_schunck(hs_img1, hs_img2, n_iters=n_iters)
    magnitude = np.hypot(u, v)
    im = ax.imshow(magnitude, cmap='viridis', vmin=0, vmax=np.hypot(*hs_true_motion))
    ax.set_title(f'{n_iters} iterations', fontsize=9)
    ax.axis('off')
plt.suptitle(f'Flow magnitude spreading outward from the square (true motion = {hs_true_motion})')
plt.tight_layout()
plt.show()

Nakon samo 1 iteracije tok je različit od nule samo neposredno uz rubove kvadrata, gdje postoje lokalne informacije o gradijentu; nakon 200 iteracija taj se sjaj proširio znatno izvan rubova u okolno ravnomjerno područje, iako još nije ni blizu svih piksela &mdash; stvarni bi postupak proveo mnogo više iteracija (ili koristio piramidu, isti postupak kojim Lucas-Kanade obrađuje velika gibanja) kako bi svugdje konvergirao. Čak i ovo djelomično širenje već pokazuje mehanizam na djelu: glatkoća prenosi informacije u područja bez vlastitog podatkovnog člana.

## Farnebäckova metoda za gusti optički tok

Globalna optimizacija metode Horn-Schunck elegantna je, ali skupa i sporo konvergira; za gusti tok (na svakom pikselu), <a href="../references.html#farneback-2003">Farnebäckova metoda (2003.)</a> &mdash; dostupna kao `cv2.calcOpticalFlowFarneback` &mdash; popularnija je metoda koja se stvarno koristi u praksi. Ostaje lokalna, poput metode Lucas-Kanade, ali pretpostavku stalnog toka unutar prozora zamjenjuje lokalnom aproksimacijom svakog susjedstva polinomom i uspoređivanjem polinomskih razvoja između kadrova.

In [ ]:
dense_motion = (5.0, -3.0)
shift_dense = np.float32([[1, 0, dense_motion[0]], [0, 1, dense_motion[1]]])
frame2_dense = cv2.warpAffine(frame1, shift_dense, (200, 200))

flow = cv2.calcOpticalFlowFarneback(frame1, frame2_dense, None, pyr_scale=0.5, levels=3,
                                     winsize=15, iterations=3, poly_n=5, poly_sigma=1.2, flags=0)

print(f'true motion: {dense_motion}')
print(f'mean flow, ALL pixels:      ({flow[..., 0].mean():.2f}, {flow[..., 1].mean():.2f})  <- biased low')

gradient_mag = np.hypot(cv2.Sobel(frame1.astype(np.float64), cv2.CV_64F, 1, 0, ksize=3),
                         cv2.Sobel(frame1.astype(np.float64), cv2.CV_64F, 0, 1, ksize=3))
textured = gradient_mag > 50
textured[:15, :] = textured[-15:, :] = textured[:, :15] = textured[:, -15:] = False  # avoid warp border artifacts

print(f'mean flow, TEXTURED pixels: ({flow[textured, 0].mean():.2f}, {flow[textured, 1].mean():.2f})  <- accurate')

To je ponovno problem otvora, u najizraženijem obliku: na ravnomjernoj pozadini bez teksture nema nikakvih lokalnih informacija za procjenu gibanja, pa je tok ondje nepouzdan i odvlači prosjek cijele slike od stvarne vrijednosti. Ograničavanje na piksele s teksturom (velikim gradijentom) daje gotovo potpuno točno gibanje.

In [ ]:
def flow_to_color(flow):
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros(flow.shape[:2] + (3,), dtype=np.uint8)
    hsv[..., 0] = angle * 180 / np.pi / 2       # hue = direction
    hsv[..., 1] = 255                            # full saturation
    hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)  # value = speed
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(frame1, cmap='gray')
axes[0].set_title('Frame 1')
axes[1].imshow(flow_to_color(flow))
axes[1].set_title(f'Dense flow field\n(color = direction, brightness = speed)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Svaki krug koji se giba daje *istu* boju jer svi ovdje imaju isto stvarno gibanje.

### Stvarni videozapis: sekvenca Army

Na stvarnom videozapisu s neovisno pokretnim objektima, ovo kodiranje bojom omogućuje trenutno razlikovanje gibanja &mdash; upravo zato predstavlja standardan način vizualizacije gustih polja toka.

Naizmjenični prikaz dvaju kadrova olakšava izravno uočavanje gibanja &mdash; promatrajte koliko se različito pomiču vojnici igračke, lopta i pozadina.

<img src="../img/army_flicker.gif" alt="Naizmjenični prikaz dvaju kadrova sekvence Army iz Middleburyja" style="display: block; margin: 0 auto; max-width: 500px;">

<p class="image-attribution">Izvor slike: <a href="https://vision.middlebury.edu/flow/data" target="_blank" rel="noopener">Middlebury Optical Flow</a></p>

In [ ]:
im1 = cv2.imread('../img/army10.png', cv2.IMREAD_GRAYSCALE)
im2 = cv2.imread('../img/army11.png', cv2.IMREAD_GRAYSCALE)
imflow = cv2.calcOpticalFlowFarneback(im1, im2, None, pyr_scale=0.5, levels=3,
                                     winsize=15, iterations=3, poly_n=5, poly_sigma=1.2, flags=0)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(im1, cmap='gray')
axes[0].set_title('Frame 1')
axes[1].imshow(flow_to_color(imflow))
axes[1].set_title(f'Dense flow field\n(color = direction, brightness = speed)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


### Zadaci

1. Ponovite usporedbu rijetkog toka Lucas-Kanade uz `large_motion = (10.0, 8.0)`. Određuje li ga `cv2.calcOpticalFlowPyrLK` i dalje točno? Kada počinje zakazivati (isprobajte sve veća gibanja) i zašto biste očekivali da piramida pomaže pomaknuti tu granicu?
2. Izmijenite sintetički prizor tako da se krugovi gibaju *različitim* brzinama (npr. polovica u jednom, a polovica u drugom smjeru). Ponovite vizualizaciju gustog Farnebäckova toka bojama i potvrdite da se dva gibanja prikazuju dvjema različitim bojama.
3. U demonstraciji problema otvora izgradite malo područje s teksturom (koje nije rub) umjesto ravnog ruba i pokažite da se iz njega mogu odrediti *obje* komponente proizvoljnog vektora gibanja &mdash; za razliku od slučaja s rubom, ovdje ne bi trebao postojati nevidljiv smjer gibanja.